In [1]:
# Create training file from the active molecules
import pandas as pd
from pathlib import Path
import json

active_molecules = pd.read_csv("../data/active_compounds_mpo.smi", sep="\t", header=None, names=["SMILES", "target"])
# Get idx2target
idx2target = json.loads(Path("../data/target_conditions_to_index_mpo.json").read_text())
idx2target = idx2target["idx_to_gene"]

# Add target to active molecules
active_molecules["target_name"] = active_molecules["target"].map(
    lambda x: idx2target[str(x)]
)
active_molecules.sample(5)

,SMILES,target,target_name
242953,O=C(NCCCCN1CCN(c2csc3cc(F)ccc23)CC1)c1cnccn1,74,ADRA1A
165895,Cc1cc(N2CCOCC2)cc2nc(-c3c(NCCn4nccc4C)ccnc3O)[...,238,IGF1R
211704,OC[C@H]1O[C@@H](n2cnc3c2NC=NC[C@H]3O)CC1O,429,BBB
62194,COC(=O)c1cc(C#N)c(NCc2ccccc2)nc1C,207,TXNRD1
179627,Cc1ccc(F)c(NC(=O)Nc2ccc(-c3ccc4nccnc4c3N)cc2)c1,92,STK17A


In [2]:
conditions = {
    "alzheimer": ["AChE", "MAOB"],
    "schizophrenia": ["D2R", "_5HT2A"],
    "parkinson": ["D2R", "D3R"],
}
for disease, targets in conditions.items():
    # Get SMILES where target_name is in targets
    smiles = active_molecules[active_molecules["target_name"].isin(targets)][["SMILES"]]
    smiles.to_csv(f"../data/training_files/active_compounds_{disease}.smi", header=False, index=False)

In [3]:
import sys
sys.path.append("..")
from benchmarks.guacamol.assess_distribution_learning import assess_distribution_learning_from_smiles

compogpt_results = {}

# Load generated molecules SMILES
for disease, targets in conditions.items():
    print(f"Disease: {disease}")
    targets_file = "_".join(targets) + "_SUM.csv"
    smiles_list = pd.read_csv(f"../generated_molecules/25-epoch/{targets_file}")["SMILES"].tolist()
    results = assess_distribution_learning_from_smiles(
        smiles_list, 
        chembl_training_file=f"../data/training_files/active_compounds_{disease}.smi",
        json_output_file=None,
        number_samples=2000
    )
    results_df = pd.DataFrame(results["results"])
    results_df["CoMPO-GPT"] = results_df["score"]
    results_df["Disease"] = disease
    results_df = results_df.set_index(["Disease", "benchmark_name"])[["CoMPO-GPT"]]
    compogpt_results[disease] = results_df
    # print(f"Disease: {disease}")
    # display(results_df)

compogpt_results_df = pd.concat(compogpt_results.values(), axis=0)
display(compogpt_results_df)

Disease: alzheimer


/home/arthurcerveira/miniconda3/envs/gnn/lib/python3.12/site-packages/fcd/fcd.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_config = torch.load(model_path)


Disease: schizophrenia
Disease: parkinson


CoMPO-GPT
Disease       benchmark_name                     
alzheimer     Validity                   0.900300
              Uniqueness                 0.875153
              Novelty                    0.973347
              KL divergence              0.888270
              Frechet ChemNet Distance   0.295305
schizophrenia Validity                   0.899200
              Uniqueness                 0.714858
              Novelty                    0.963130
              KL divergence              0.920486
              Frechet ChemNet Distance   0.307643
parkinson     Validity                   0.899000
              Uniqueness                 0.704116
              Novelty                    0.957820
              KL divergence              0.927711
              Frechet ChemNet Distance   0.260788

In [4]:
conditions = {
    "alzheimer": ["AChE", "MAOB"],
    "schizophrenia": ["D2R", "_5HT2A"],
    "parkinson": ["D2R", "D3R"],
}
assays_dir = Path("..") / "data" / "Assays-pXC50"

for disease, targets in conditions.items():
    active_molecules_disease = pd.DataFrame()

    for target in targets:
        assay_path = assays_dir / (target + ".csv")
        assay_df = pd.read_csv(assay_path)
        assay_df["pXC50"] = pd.to_numeric(assay_df["pXC50"], errors="coerce")
        assay_df = assay_df.dropna(how="any")
        active_smiles = assay_df[assay_df["pXC50"] >= 6.0]["SMILES"]
        active_molecules_disease = pd.concat([active_molecules_disease, active_smiles])
    
    active_molecules_disease.to_csv(
        f"../data/training_files/all_active_compounds_{disease}.smi", 
        header=False, index=False
    )

In [5]:
# baselines = ['polygon-mma', 'polygon-le', 'MTMol-GPT', 'DeepLig']
# baselines = ['POLYGON-MMA', 'polygon-le', 'MTMol-GPT', 'DeepLig']
baselines = ['POLYGON', 'MTMol-GPT', 'DeepLig']
baselines_results = {}

for disease, targets in conditions.items():
    baselines_disease = pd.DataFrame()

    for baseline in baselines:
        print(f"Disease: {disease}, Baseline: {baseline}")
        df = pd.read_csv(f"../generated_molecules/{baseline}/{disease}.csv", header=None)[0]
        active_smiles = pd.read_csv(f"../data/training_files/all_active_compounds_{disease}.smi", header=None)[0]
        num_active_mols = len(active_smiles)
        print(f"Number of active molecules: {num_active_mols}")
        results = assess_distribution_learning_from_smiles(
            df.tolist(), 
            chembl_training_file=f"../data/training_files/all_active_compounds_{disease}.smi",
            json_output_file=None,
            number_samples=min(num_active_mols, 10_000)
        )

        results_df = pd.DataFrame(results["results"])
        # View score as 4 decimal places
        results_df["score"] = results_df["score"].round(6)
        results_df[baseline] = results_df["score"]
        results_df["Disease"] = disease
        results_df = results_df.set_index(["Disease", "benchmark_name"])[[baseline]]
        if baselines_disease.empty:
            baselines_disease = results_df
            continue

        baselines_disease = baselines_disease.merge(results_df, left_index=True, right_index=True, how="right")
        # display(baselines_disease)

    baselines_results[disease] = baselines_disease

baselines_results_df = pd.concat(baselines_results.values(), axis=0)
display(baselines_results_df)

Disease: alzheimer, Baseline: POLYGON
Number of active molecules: 2764
Disease: alzheimer, Baseline: MTMol-GPT
Number of active molecules: 2764
Disease: alzheimer, Baseline: DeepLig
Number of active molecules: 2764
Disease: schizophrenia, Baseline: POLYGON
Number of active molecules: 8643
Disease: schizophrenia, Baseline: MTMol-GPT
Number of active molecules: 8643
Disease: schizophrenia, Baseline: DeepLig
Number of active molecules: 8643
Disease: parkinson, Baseline: POLYGON
Number of active molecules: 8158
Disease: parkinson, Baseline: MTMol-GPT
Number of active molecules: 8158
Disease: parkinson, Baseline: DeepLig
Number of active molecules: 8158


POLYGON  MTMol-GPT   DeepLig
Disease       benchmark_name                                         
alzheimer     Validity                  0.975300   0.969524  0.967093
              Uniqueness                0.413616   0.321316  0.651877
              Novelty                   1.000000   0.454295  1.000000
              KL divergence             0.003643   0.991088  0.230007
              Frechet ChemNet Distance  0.000010   0.736994  0.000003
schizophrenia Validity                  0.998200   0.976095  0.929493
              Uniqueness                0.372971   0.507659  0.549494
              Novelty                   1.000000   0.323852  1.000000
              KL divergence             0.001436   0.987935  0.222160
              Frechet ChemNet Distance  0.000001   0.900377  0.000012
parkinson     Validity                  0.947700   0.972667  0.994899
              Uniqueness                0.443600   0.469989  0.013270
              Novelty                   1.000000   0.370208  1.000000
              KL divergence             0.062345   0.988119  0.084845
              Frechet ChemNet Distance  0.000043   0.906446  0.000025

In [6]:
baselines_compogpt = compogpt_results_df.merge(baselines_results_df, left_index=True, right_index=True, how="outer")
baselines_compogpt.style \
    .background_gradient(cmap="Blues", axis=1) \
    .format("{:.4f}") \
    .set_properties(**{'font-size': '12pt'}) \
    .set_table_styles([
        {'selector': 'th', 'props': [('line-height', '1.5')]},
        {'selector': 'td', 'props': [('line-height', '1.5')]}
    ])

In [7]:
baselines_compogpt = compogpt_results_df.merge(baselines_results_df, left_index=True, right_index=True, how="outer")
baselines_compogpt.style \
    .background_gradient(cmap="Blues", axis=1) \
    .format("{:.4f}") \
    .set_properties(**{'font-size': '12pt'}) \
    .set_table_styles([
        {'selector': 'th', 'props': [('line-height', '1.5')]},
        {'selector': 'td', 'props': [('line-height', '1.5')]}
    ])

In [8]:
print(baselines_compogpt.to_latex(float_format="%.3f"))

\begin{tabular}{llrrrr}
\toprule
 &  & CoMPO-GPT & POLYGON & MTMol-GPT & DeepLig \\
Disease & benchmark_name &  &  &  &  \\
\midrule
\multirow[t]{5}{*}{alzheimer} & Validity & 0.900 & 0.975 & 0.976 & 0.967 \\
 & Uniqueness & 0.875 & 0.414 & 0.508 & 0.652 \\
 & Novelty & 0.973 & 1.000 & 0.999 & 1.000 \\
 & KL divergence & 0.888 & 0.004 & 0.868 & 0.230 \\
 & Frechet ChemNet Distance & 0.295 & 0.000 & 0.016 & 0.000 \\
\cline{1-6}
\multirow[t]{5}{*}{schizophrenia} & Validity & 0.899 & 0.998 & 0.976 & 0.929 \\
 & Uniqueness & 0.715 & 0.373 & 0.508 & 0.549 \\
 & Novelty & 0.963 & 1.000 & 0.324 & 1.000 \\
 & KL divergence & 0.920 & 0.001 & 0.988 & 0.222 \\
 & Frechet ChemNet Distance & 0.308 & 0.000 & 0.900 & 0.000 \\
\cline{1-6}
\multirow[t]{5}{*}{parkinson} & Validity & 0.899 & 0.948 & 0.973 & 0.995 \\
 & Uniqueness & 0.704 & 0.444 & 0.470 & 0.013 \\
 & Novelty & 0.958 & 1.000 & 0.370 & 1.000 \\
 & KL divergence & 0.928 & 0.062 & 0.988 & 0.085 \\
 & Frechet ChemNet Distance & 0.261 & 0.000 